In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('data/premium.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
# 중복체크
df[df.duplicated(keep=False)]

,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [5]:
df = df.drop_duplicates()
df.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   sex       1337 non-null   str    
 2   bmi       1332 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   str    
 5   region    1337 non-null   str    
 6   charges   1337 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 83.6 KB


In [6]:
#bmi의 널처리
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder

y = df['charges']
X = df.drop('charges', axis=1)

# X = df.iloc[:, :-1]
# y = df.iloc[:, -1]

# 문자열 데이터의 수치화 > LabelEncoder
# 1. Dictionary Mapping 이용 : Label 지정
mapping = {
    'sex': {'female': 0, 'male': 1},
    'smoker': {'no': 0, 'yes': 1}
}

for col, m in mapping.items():
    df[col] = df[col].str.strip() # 공백 삭제
    X[col] = df[col].map(m)  

# region만 따로 처리 (순서가 중요하지 않다면)
X['region'] = LabelEncoder().fit_transform(df['region'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling for better performance : Only for Quanitity Attributes
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled['bmi'] = scaler.fit_transform(X_train[['bmi']]) # DataFrame 으로 보이게
X_test_scaled['bmi'] = scaler.transform(X_test[['bmi']])

NameError: name 'df' is not defined

In [63]:
X_train_scaled.head()

,age,sex,bmi,children,smoker,region
1114,23,1,-0.999446,0,0,0
968,21,1,-0.794559,2,0,0
599,52,0,1.159756,2,0,1
170,63,1,1.814235,0,0,2
275,47,0,-0.652713,2,0,0


# 모델별 학습 및 평가

# 선형회귀모델

In [64]:
from sklearn.linear_model import LinearRegression

# 선형회귀모델 : 정의 및 학습
model_linear = LinearRegression()
# model_linear.fit(X_train, y_train)
model_linear.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [58]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def get_model_performance(real, pred) -> str :
    
    rmse = np.sqrt(mean_squared_error(real, pred))
    r2 = r2_score(real, pred)
    
    print(f"R-Square Score : {r2:.3f}")
    print(f"Real(Test) Value('charges') Mean : {real.mean():,.1f}, Mean-Square-Root Error : {rmse:,.1f}")
    
    return r2, rmse 

In [65]:
# y_pred = model_linear.predict(X_test)
y_pred = model_linear.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

r2, rmse = get_model_performance(y_test, y_pred)

mae, float(rmse), r2

R-Square Score : 0.807
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 5,962.4


(4170.6423560015255, 5962.393019338352, 0.8065362865570331)

# 다항회귀모델  


In [66]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# features = PolynomialFeatures(degree=2, include_bias=False) # 2차 항까지 사용
# model_poly = Pipeline([('poly', features), ('linear', LinearRegression())])

# model_poly.fit(X_train, y_train)

for degree in [2,3,4] :
    features = PolynomialFeatures(degree=degree, include_bias=False)
    model_poly = Pipeline([('poly', features), ('linear', LinearRegression())])
    
    model_poly.fit(X_train_scaled, y_train)
    poly_pred = model_poly.predict(X_test_scaled)
    
    print("Performance at DEGREE :", degree)
    get_model_performance(y_test, poly_pred)
    print("-"*100)


Performance at DEGREE : 2
R-Square Score : 0.886
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 4,569.4
----------------------------------------------------------------------------------------------------
Performance at DEGREE : 3
R-Square Score : 0.877
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 4,745.7
----------------------------------------------------------------------------------------------------
Performance at DEGREE : 4
R-Square Score : 0.855
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 5,166.3
----------------------------------------------------------------------------------------------------


** 2차항에서 가장 성능이 높게 나타남 : 차수가 높아질수록 과적합 발생

# Random Forest

In [71]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV


model_rfr = RandomForestRegressor(random_state=42, n_jobs=-1)

# 학습용 문제와 정답을 주고 모델 학습시키기
param_grid = {
    'n_estimators': [100, 200, 300],         # 트리 개수 (많을수록 안정적이나 300 이상은 효율 저하 가능)
    'max_depth': [None, 10, 20, 30],         # 트리 최대 깊이 (None은 끝까지 분할, 10~20은 과적합 방지용)
    'min_samples_split': [2, 5, 10],         # 노드 분할을 위한 최소 샘플 수 (데이터 1% 내외인 10까지 권장)
    'min_samples_leaf': [1, 2, 4],           # 리프 노드가 되기 위한 최소 샘플 수
    'max_features': ['sqrt', 'log2', None]   # 각 노드 분할 시 고려할 특성 수 (None은 전체 특성 사용)
}

# 3. GridSearchCV 객체 생성 (5-fold 교차 검증 사용)
grid_search = GridSearchCV(model_rfr, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)

# 4. 학습 (X_train, y_train은 앞서 변환한 NumPy 배열)
grid_search.fit(X_train_scaled, y_train)

# 5. 최적의 결과 확인
print(f"최적 파라미터: {grid_search.best_params_}")
print(f"최고 정확도: {grid_search.best_score_:.4f}")

# 6. 최적의 모델 추출
model_rfr = grid_search.best_estimator_

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
최적 파라미터: {'max_depth': 10, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
최고 정확도: 0.8297


In [52]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV


model_rfr2 = RandomForestRegressor(random_state=42, n_jobs=-1)

# 학습용 문제와 정답을 주고 모델 학습시키기 # 2차 수정
param_grid2 = {
    'n_estimators': [280, 300, 320],         # 트리 개수 (많을수록 안정적이나 300 이상은 효율 저하 가능)
    'max_depth': [8, 10, 12],         # 트리 최대 깊이 (None은 끝까지 분할, 10~20은 과적합 방지용)
    'min_samples_split': [7, 10, 13],         # 노드 분할을 위한 최소 샘플 수 (데이터 1% 내외인 10까지 권장)
    'min_samples_leaf': [4] ,           # 리프 노드가 되기 위한 최소 샘플 수
    'max_features': ['sqrt', 'log2', None]   # 각 노드 분할 시 고려할 특성 수 (None은 전체 특성 사용)
}

# 3. GridSearchCV 객체 생성 (5-fold 교차 검증 사용)
grid_search2 = GridSearchCV(model_rfr2, param_grid2, cv=5, scoring='r2', n_jobs=-1, verbose=1)

# 4. 학습 (X_train, y_train은 앞서 변환한 NumPy 배열)
grid_search2.fit(X_train_scaled, y_train)

# 5. 최적의 결과 확인
print(f"최적 파라미터: {grid_search2.best_params_}")
print(f"최고 정확도: {grid_search2.best_score_:.4f}")

# 6. 최적의 모델 추출
model_rfr2 = grid_search2.best_estimator_

Fitting 5 folds for each of 81 candidates, totalling 405 fits
최적 파라미터: {'max_depth': 8, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 13, 'n_estimators': 320}
최고 정확도: 0.8318


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# ==========================================
# 학습 후 예측 및 평가
# ==========================================
# 학습에 사용하지 않은 테스트용 문제(X_test)를 주고 예측해보기
model_rfr.fit(X_train_scaled, y_train)
y_pred = model_rfr.predict(X_test_scaled)

# 실제 정답(y_test)과 모델이 예측한 값(y_pred)을 비교하여 정확도 계산
r2, rmse = get_model_performance(y_test, y_pred)


model_rfr2.fit(X_train_scaled, y_train)
y_pred2 = model_rfr2.predict(X_test_scaled)

# 실제 정답(y_test)과 모델이 예측한 값(y_pred)을 비교하여 정확도 계산
r2, rmse = get_model_performance(y_test, y_pred2)

R-Square Score : 0.899
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 4,316.6
R-Square Score : 0.900
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 4,285.2


In [73]:
model_rfr2.feature_importances_, df.columns

s = pd.Series(model_rfr2.feature_importances_, index=df.columns.drop(df.columns[-1]))
s

age         0.130443
sex         0.002242
bmi         0.181143
children    0.013926
smoker      0.665655
region      0.006591
dtype: float64

# XGBRegressor

In [74]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(n_estimators=100, random_state=0)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_xgb)
mse = mean_squared_error(y_test, y_pred_xgb)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_xgb)
mae, mse, rmse, r2

(3026.6952938129884,
 25548063.10037025,
 np.float64(5054.509184913037),
 0.8609675101951677)

# Parameter Optimization

In [87]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'random_state': 0
    }
    model = RandomForestRegressor(**params)
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2').mean()
    # score = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f'Best R2 : {round(study.best_value, 4)}')
print(f'Best Params : {study.best_params}')

[I 2026-03-03 12:05:47,887] A new study created in memory with name: no-name-8001219e-0a20-4b34-b93f-5460c97f87f1
[I 2026-03-03 12:05:48,613] Trial 0 finished with value: 0.8375482729406947 and parameters: {'n_estimators': 149, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.8375482729406947.
[I 2026-03-03 12:05:49,714] Trial 1 finished with value: 0.8304322718663165 and parameters: {'n_estimators': 189, 'max_depth': 8, 'min_samples_split': 8, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.8375482729406947.
[I 2026-03-03 12:05:50,530] Trial 2 finished with value: 0.8353606053294683 and parameters: {'n_estimators': 154, 'max_depth': 6, 'min_samples_split': 9, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.8375482729406947.
[I 2026-03-03 12:05:51,370] Trial 3 finished with value: 0.8351863117777624 and parameters: {'n_estimators': 169, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.

Best R2 : 0.8376
Best Params : {'n_estimators': 63, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 3}


In [93]:
best_model = RandomForestRegressor(**study.best_params)
best_model.fit(X_train_scaled, y_train)

y_pred_best = best_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred_best)
mse = mean_squared_error(y_test, y_pred_best)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_best)

print(f'RandomForest (Optuna) - MAE: {round(mae, 4)} / MSE: {round(mse, 4)}')
print(f'RandomForest (Optuna) - rmse: {round(rmse, 4)} / r2: {round(r2, 4)}')

RandomForest (Optuna) - MAE: 2469.2125 / MSE: 17752582.3134
RandomForest (Optuna) - rmse: 4213.3813 / r2: 0.9034
